In [27]:
import numpy as np
import string 

sanitizer = lambda x: x.lower().translate(str.maketrans('', '', string.punctuation)).strip().replace("—", " ") #patchwork ass sanitization function that removes leading spaces, puncutation and replaces - with a space
clean_script_list = np.loadtxt("../data/hobbit1.csv", delimiter=',', usecols = 1,  skiprows = 1, dtype=str, converters = sanitizer, encoding="utf-8")

tokenized_script = [sentence.split() for sentence in clean_script_list] #split into 2d array of sentences split into words

vocab = sorted(set(word for sentence in tokenized_script for word in sentence)) #get a sorted set of unique words
word_to_index = {word: idx for idx, word in enumerate(vocab)} #initialize a dict, word corresponds to a value index

def one_hot_generator(word): 
    one_hot = np.zeros(len(vocab))
    one_hot[word_to_index[word]] = 1 
    return one_hot




In [78]:
#need a user specified window-size, hardcode this to 3 for now 
#CBOW IMPLEMENTATION
WINDOW_SIZE = 3

def CBOW_generate_training_data(tokenized_sen, word_dict, window_size = 1):
    X_train = []
    y_train = []

    for sen in tokenized_sen: 
        aggregated_cv = np.zeros(len(vocab))
        for i, target in enumerate(sen): 

            aggregated_cv = np.zeros(len(vocab))
            start = max(0, i - window_size)
            end = min(len(sen), i + window_size)

            cv = [neighbor for neighbor in sen[start:end + 1] if neighbor != target]
            for v in cv: 
                aggregated_cv += one_hot_generator(v)

            y_train.append(one_hot_generator(target))
            X_train.append(aggregated_cv)

    return np.reshape(np.array(X_train), (-1, len(vocab))), np.reshape(np.array(y_train), (-1, 1))

X_train, y_train = CBOW_generate_training_data(tokenized_script, word_to_index, 5)
X_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(4815, 1292))

In [ ]:
def softmax(x): 
    e_x = np.exp(x)
    return e_x / np.sum(e_x)

def forward_prop(A0, W0, b0, W1, b1):
    A1 = W0 @ A0 + b0
    z1 = W1 @ A1 + b1
    A3 = softmax(z1)
    return A3

def init_vals(): 
    W0 = np.random.randn(EMBEDDING_DIM, len(vocab)) #hidden layer 
    b0 = np.random.randn(EMBEDDING_DIM, 1)

    W1 = np.random.randn(len(vocab), EMBEDDING_DIM)
    b1 = np.random.randn(len(vocab), 1)

    return W0, b0, W1, b1



EMBEDDING_DIM = 100

W0, b0, W1, b1 = init_vals()

x = np.reshape(X_train[0], (-1, 1))
model_output = forward_prop(x, W0, b0, W1, b1)
loss = np.square(x - model_output)




array([[-9.61224575e-20],
       [-4.24323418e-33],
       [-1.25417905e-31],
       ...,
       [-2.97780121e-32],
       [-1.51106882e-43],
       [-1.83999661e-22]], shape=(1292, 1))